# 02 — Prétraitement et encodage des données (version corrigée)
### Mini-projet Data Mining — Classification du Score de Crédit · Groupe RSD10

**Objectif :** partir de `data/raw/train.csv`, nettoyer les anomalies de format, séparer les **clients** entre entraînement et validation, traiter les valeurs manquantes, encoder les catégories et standardiser les variables nécessaires.

**Corrections par rapport à la version précédente :**
1. Séparation **par `Customer_ID`** (et non `train_test_split` ligne par ligne) ; aucun client commun entre les ensembles.
2. Lecture reproductible du CSV brut, sans dépendre d'un `train_clean.csv` créé ailleurs.
3. Les imputations, encodages appris et standardisation sont **ajustés sur l'entraînement uniquement**.
4. Conservation d'un objet `preprocessor` (`ColumnTransformer`) **réajustable dans chaque pli de validation croisée** : on ne fait pas de validation croisée sur des données déjà imputées/standardisées globalement.
5. Aucune sortie n'affiche `Name` ou `SSN`. Les fichiers générés restent dans `data/processed/`, qui ne doit pas être publié sans contrôle.

**Important :** le notebook produit les **entrées du futur modèle**, pas un modèle de prédiction entraîné. Il ne modifie pas `data/raw/train.csv`.

## 1. Bibliothèques et localisation du dataset
Dans VS Code, sélectionner le **kernel du projet**. Les bibliothèques doivent être installées dans ce kernel, pas dans un autre Python. Si nécessaire, depuis le terminal du projet :

```powershell
.\.venv\Scripts\python.exe -m pip install pandas numpy scikit-learn joblib ipykernel
```

In [ ]:
import sys
import re
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import sklearn
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("Interpréteur Python :", sys.executable)
print("Version scikit-learn :", sklearn.__version__)

# Fonctionne depuis la racine du dépôt ou depuis notebooks/
racine = next(
    (d for d in [Path.cwd(), *Path.cwd().parents]
     if (d / 'data' / 'raw' / 'train.csv').is_file()),
    None
)
if racine is None:
    raise FileNotFoundError("Fichier data/raw/train.csv introuvable : ouvrir le projet dans VS Code.")

fichier_brut = racine / 'data' / 'raw' / 'train.csv'
df_brut = pd.read_csv(fichier_brut, low_memory=False)

obligatoires = {'Customer_ID', 'Credit_Score', 'Age', 'Num_of_Loan', 'Type_of_Loan'}
manquantes = obligatoires - set(df_brut.columns)
if manquantes:
    raise ValueError(f"Colonnes obligatoires absentes : {sorted(manquantes)}")
if df_brut['Customer_ID'].isna().any() or df_brut['Credit_Score'].isna().any():
    raise ValueError('Customer_ID et Credit_Score doivent être renseignés avant la division.')

print("Dimensions du dataset brut :", df_brut.shape)
print("Nombre de clients distincts :", df_brut['Customer_ID'].nunique())
print("Catégories du score :")
display(df_brut['Credit_Score'].value_counts().rename('effectif').to_frame())

## 2. Contrôle qualité initial et règles de nettoyage

Les valeurs comme `24_` doivent redevenir numériques ; les marqueurs `_`, `_______`, `!@9#%8` signifient *information absente*. Le marqueur `__10000__` dans certaines colonnes est traité comme un code corrompu, **pas** comme un montant réel.

Les âges ≤ 0 ou > 120 et les nombres de prêts < 0 ou > 100 sont traités comme manquants. Le seuil de **100 prêts reste un choix exploratoire**, à discuter dans le rapport. Les revenus très élevés ne sont pas supprimés automatiquement.

Seules des **règles déterministes ligne par ligne** sont appliquées dans cette étape : aucune médiane, fréquence, moyenne ou catégorie n'est calculée sur le jeu complet.

In [ ]:
colonnes_numeriques = [
    'Age', 'Annual_Income', 'Monthly_Inhand_Salary',
    'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate',
    'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment',
    'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Outstanding_Debt',
    'Credit_Utilization_Ratio', 'Credit_History_Age',
    'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance'
]

colonnes_categorielles = [
    'Occupation', 'Payment_of_Min_Amount', 'Payment_Behaviour',
    'Credit_Mix', 'Type_of_Loan', 'Month'
]

attendues = set(colonnes_numeriques + colonnes_categorielles)
if attendues - set(df_brut.columns):
    raise ValueError(f"Variables d'entrée absentes : {sorted(attendues - set(df_brut.columns))}")

# 17 colonnes numériques + 6 catégorielles = 23 variables candidates,
# après exclusion des 4 identifiants et de la cible.
print('Variables numériques :', len(colonnes_numeriques))
print('Variables catégorielles :', len(colonnes_categorielles))

manquants_bruts = df_brut.isna().sum().sort_values(ascending=False)
display(manquants_bruts[manquants_bruts > 0].rename('cellules vides').to_frame())

In [ ]:
def convertir_historique_credit(serie):
    """Convertit '22 Years and 1 Months' en mois ; accepte aussi les valeurs déjà numériques."""
    t = serie.astype('string').str.strip()
    deja_numerique = pd.to_numeric(t, errors='coerce')
    annees = pd.to_numeric(t.str.extract(r'(?i)(\d+)\s*years?', expand=False), errors='coerce')
    mois = pd.to_numeric(t.str.extract(r'(?i)(\d+)\s*months?', expand=False), errors='coerce')
    en_mois = 12 * annees.fillna(0) + mois.fillna(0)
    en_mois = en_mois.where(annees.notna() | mois.notna(), np.nan)
    return deja_numerique.fillna(en_mois).astype(float)


def nettoyer_valeurs_brutes(table):
    """Nettoyage déterministe, applicable à des lignes nouvelles sans apprentissage statistique."""
    resultat = table.loc[:, colonnes_numeriques + colonnes_categorielles].copy()

    for col in colonnes_numeriques:
        if col == 'Credit_History_Age':
            resultat[col] = convertir_historique_credit(table[col])
            continue

        txt = table[col].astype('string').str.strip()
        # Les doubles underscores encadrant un nombre sont un marqueur corrompu.
        marque_corrompue = txt.str.startswith('__').fillna(False)
        txt = txt.mask(marque_corrompue)
        txt = txt.replace({'': pd.NA, '_': pd.NA, '_______': pd.NA, '!@9#%8': pd.NA})
        txt = txt.str.replace(r'_+$', '', regex=True)
        resultat[col] = pd.to_numeric(txt, errors='coerce').astype(float)

    # On n'utilise pas d'estimation pour corriger ces anomalies : elles deviennent NaN.
    resultat.loc[~resultat['Age'].between(1, 120), 'Age'] = np.nan
    resultat.loc[~resultat['Num_of_Loan'].between(0, 100), 'Num_of_Loan'] = np.nan
    resultat.loc[resultat['Num_of_Delayed_Payment'] < 0, 'Num_of_Delayed_Payment'] = np.nan

    for col in colonnes_categorielles:
        txt = table[col].astype('string').str.strip()
        txt = txt.replace({'': pd.NA, '_': pd.NA, '_______': pd.NA, '!@9#%8': pd.NA})
        # Les valeurs de Type_of_Loan seront séparées en modalités à l'étape 4.
        resultat[col] = txt.astype(object).where(txt.notna(), np.nan)

    return resultat

## 3. Division entraînement / validation **par client**

Les **12 500 clients** du jeu complet (à vérifier ci-dessus) ont généralement plusieurs mois d'observations. Une division ligne par ligne pourrait placer un même client dans l'entraînement et dans la validation, ce qui fausserait l'évaluation.

On réserve environ **20 % des clients** pour une validation finale. Cette validation devra rester intacte pendant le choix des modèles et leurs hyperparamètres. `GroupShuffleSplit` garantit des clients distincts, mais **ne garantit pas parfaitement la proportion des classes** : nous la contrôlons après la division.

In [ ]:
separateur = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
idx_train, idx_val = next(separateur.split(df_brut, df_brut['Credit_Score'], df_brut['Customer_ID']))

lignes_train = df_brut.iloc[idx_train].reset_index(drop=True)
lignes_val = df_brut.iloc[idx_val].reset_index(drop=True)

groups_train = lignes_train['Customer_ID'].reset_index(drop=True)
groups_val = lignes_val['Customer_ID'].reset_index(drop=True)
y_train = lignes_train['Credit_Score'].reset_index(drop=True)
y_val = lignes_val['Credit_Score'].reset_index(drop=True)

clients_communs = set(groups_train) & set(groups_val)
print('Clients communs entre entraînement et validation :', len(clients_communs))
assert not clients_communs, 'Fuite de clients détectée !'
print('Clients entraînement :', groups_train.nunique())
print('Clients validation   :', groups_val.nunique())
print('Lignes entraînement :', len(lignes_train))
print('Lignes validation   :', len(lignes_val))

repartition = pd.concat({
    'Entraînement (%)': y_train.value_counts(normalize=True).mul(100),
    'Validation (%)': y_val.value_counts(normalize=True).mul(100)
}, axis=1)
display(repartition.round(2))
assert set(y_val.unique()).issubset(set(y_train.unique()))

## 4. Préparation des variables et encodages déterministes

On nettoie **séparément** les lignes d'entraînement et de validation. Les corrections appliquées ci-dessous sont déterministes ; en revanche, le vocabulaire des **types de prêts** est construit **uniquement avec l'entraînement**.

Nous conservons les choix pertinents de l'ancien notebook :
- mois : encodage cyclique `sin/cos` ;
- `Type_of_Loan` : encodage **multi-hot** (plusieurs prêts possibles par client) ;
- `Credit_Mix` : encodage ordinal (*Bad*, *Standard*, *Good*, inconnu) ;
- `Occupation`, `Payment_of_Min_Amount`, `Payment_Behaviour` : **one-hot** appris uniquement sur le train, à l'étape suivante.

**Limite à documenter :** traiter `Credit_Mix` inconnu comme `-1` suppose une distance numérique artificielle. Une alternative à tester en phase 3 est le one-hot pour `Credit_Mix`.

In [ ]:
base_train = nettoyer_valeurs_brutes(lignes_train).reset_index(drop=True)
base_val = nettoyer_valeurs_brutes(lignes_val).reset_index(drop=True)

print('Valeurs manquantes ou invalides après nettoyage déterministe :')
resume_qualite = pd.DataFrame({
    'train': base_train.isna().sum(),
    'validation': base_val.isna().sum()
}).sort_values('train', ascending=False)
display(resume_qualite.loc[(resume_qualite != 0).any(axis=1)])

print('Âges invalides restants (hors NaN) :', int((~base_train['Age'].dropna().between(1, 120)).sum()))
print('Prêts invalides restants (hors NaN) :', int((~base_train['Num_of_Loan'].dropna().between(0, 100)).sum()))

In [ ]:
mois_vers_numero = {
    'January':1, 'February':2, 'March':3, 'April':4,
    'May':5, 'June':6, 'July':7, 'August':8,
    'September':9, 'October':10, 'November':11, 'December':12
}


def decomposer_prets(valeur):
    if pd.isna(valeur):
        return []
    # Les listes peuvent contenir « and » avant le dernier type de prêt.
    parties = str(valeur).split(',')
    resultat = []
    for p in parties:
        nom = re.sub(r'^and\s+', '', p.strip(), flags=re.IGNORECASE).strip()
        if nom:
            resultat.append(nom)
    return resultat

# Le vocabulaire vient uniquement de l'entraînement.
loan_types = sorted({
    nom for ligne in base_train['Type_of_Loan']
    for nom in decomposer_prets(ligne)
})
print('Types de prêts appris sur entraînement (nombre) :', len(loan_types))
print(loan_types)

In [ ]:
def construire_variables(base, vocabulaire_prets):
    x = base[colonnes_numeriques + ['Occupation', 'Payment_of_Min_Amount', 'Payment_Behaviour']].copy()

    numero_mois = base['Month'].map(mois_vers_numero)
    x['Month_sin'] = np.sin(2 * np.pi * (numero_mois - 1) / 12)
    x['Month_cos'] = np.cos(2 * np.pi * (numero_mois - 1) / 12)

    # Ordinal choisi dans le notebook précédent ; code -1 pour information absente/inconnue.
    x['Credit_Mix_ordinal'] = (
        base['Credit_Mix'].map({'Bad': 0, 'Standard': 1, 'Good': 2})
                          .fillna(-1).astype(float)
    )

    listes = base['Type_of_Loan'].map(decomposer_prets)
    # Toutes les colonnes sont définies par le vocabulaire TRAIN.
    for nom in vocabulaire_prets:
        x['Loan_' + nom.replace(' ', '_').replace('-', '_')] = listes.map(lambda l: int(nom in l)).astype(float)
    x['Loan_Inconnu'] = base['Type_of_Loan'].isna().astype(float)

    # Quand la validation contient une catégorie complètement nouvelle,
    # elle ne crée PAS de nouvelle colonne.
    x.index = range(len(x))
    return x

X_train_features = construire_variables(base_train, loan_types)
X_val_features = construire_variables(base_val, loan_types)
assert X_train_features.columns.tolist() == X_val_features.columns.tolist()
print('Dimensions avant imputation et one-hot :', X_train_features.shape, X_val_features.shape)
print('Colonnes de travail :', len(X_train_features.columns))

## 5. Imputation, One-Hot Encoding et StandardScaler **sans fuite de données**

Nous définissons un `ColumnTransformer` scikit-learn :
- **Numériques :** imputation par **médiane** → `StandardScaler`.
- **Catégorielles :** imputation par la constante **« Inconnu »** → `OneHotEncoder(handle_unknown='ignore')`.
- **Variables construites :** mois cycliques, prêts multi-hot et crédit ordinal, transmis sans standardisation.

La **seule instruction `fit`** porte sur `X_train_features`. On utilise uniquement `transform` sur `X_val_features`. Le `ColumnTransformer` sera aussi **réajustable par pli** dans la validation croisée de la phase 3.

In [ ]:
colonnes_categories_onehot = ['Occupation', 'Payment_of_Min_Amount', 'Payment_Behaviour']
colonnes_construites = [
    c for c in X_train_features.columns
    if c not in colonnes_numeriques + colonnes_categories_onehot
]

numerical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('standardisation', StandardScaler())
])

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Inconnu')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipe, colonnes_numeriques),
        ('cat', categorical_pipe, colonnes_categories_onehot),
        ('autres', 'passthrough', colonnes_construites)
    ],
    remainder='drop',
    sparse_threshold=0,
    verbose_feature_names_out=False
).set_output(transform='pandas')

X_train_preprocessed = preprocessor.fit_transform(X_train_features)
X_val_preprocessed = preprocessor.transform(X_val_features)

print('Dimensions après prétraitement :')
print('Entraînement :', X_train_preprocessed.shape)
print('Validation   :', X_val_preprocessed.shape)
print('Valeurs manquantes :', int(X_train_preprocessed.isna().sum().sum()),
      int(X_val_preprocessed.isna().sum().sum()))
assert X_train_preprocessed.columns.tolist() == X_val_preprocessed.columns.tolist()
assert X_train_preprocessed.isna().sum().sum() == 0
assert X_val_preprocessed.isna().sum().sum() == 0
assert np.isfinite(X_train_preprocessed.to_numpy(dtype=float)).all()
assert np.isfinite(X_val_preprocessed.to_numpy(dtype=float)).all()
print('Toutes les colonnes sont numériques et finies.')

## 6. Vérification de la cohérence du prétraitement

Les variables numériques standardisées ont approximativement une moyenne 0 et un écart-type 1 **sur l'entraînement** ; ce n'est pas nécessairement le cas sur la validation. Les colonnes binaires et les encodages cycliques ne sont pas standardisés.

**Attention :** les données `X_train_preprocessed` servent à **inspecter** le prétraitement ou à entraîner un modèle final *une fois les choix fixés*. Pour la **validation croisée** et `GridSearchCV`, il faut repartir de `X_train_features` et laisser le pipeline réajuster l'imputation et le scaler **à chaque pli**.

In [ ]:
print('Aperçu des statistiques du train normalisé :')
display(X_train_preprocessed[colonnes_numeriques].agg(['mean', 'std', 'min', 'max']).T.round(3))
print('Nombre total de colonnes :', X_train_preprocessed.shape[1])
print('Colonnes correspondant aux types de prêts :', sum(c.startswith('Loan_') for c in X_train_features.columns))

# Vérifier le respect des groupes durant les 5 plis prévus pour la phase 3.
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
for pli, (i_train, i_test) in enumerate(cv.split(X_train_features, y_train, groups_train), 1):
    assert not set(groups_train.iloc[i_train]) & set(groups_train.iloc[i_test])
    print(f'Pli {pli} : {len(i_train)} entraînement, {len(i_test)} validation, clients communs = 0')

## 7. Export contrôlé vers `data/processed/`

Les fichiers de données **préparées mais non imputées globalement** sont destinés aux expérimentations de la phase 3 ; ils conservent les valeurs manquantes pour qu'une validation croisée puisse apprendre les médianes **dans chaque pli**. Nous sauvegardons également les étiquettes `y` et les groupes `Customer_ID` pour la validation croisée.

Le fichier `preprocessor.joblib` contient l'encodeur/imputeur entraîné sur **l'entraînement seulement**. Il s'applique à une table **déjà passée dans les fonctions de nettoyage et de construction des variables** de ce notebook ; ces fonctions devront être déplacées plus tard dans `src/preprocessing.py` pour construire l'application Streamlit. Ne pas charger un fichier `.joblib` provenant d'une source non fiable.

**Ne pas publier `data/raw/`, `data/processed/` ou des identifiants clients sur un GitHub public sans examen et anonymisation adaptés.** Le professeur n'exige pas de publier ces données, mais demande une méthode claire pour les obtenir.

In [ ]:
dossier_sortie = racine / 'data' / 'processed'
dossier_sortie.mkdir(parents=True, exist_ok=True)

# Important : ne pas exporter de Name, SSN ou ID.
X_train_features.to_csv(dossier_sortie / 'X_train_features.csv', index=False)
X_val_features.to_csv(dossier_sortie / 'X_val_features.csv', index=False)
y_train.rename('Credit_Score').to_csv(dossier_sortie / 'y_train.csv', index=False)
y_val.rename('Credit_Score').to_csv(dossier_sortie / 'y_val.csv', index=False)
groups_train.rename('Customer_ID').to_csv(dossier_sortie / 'groups_train.csv', index=False)
groups_val.rename('Customer_ID').to_csv(dossier_sortie / 'groups_val.csv', index=False)
joblib.dump(preprocessor, dossier_sortie / 'preprocessor.joblib')

# Ces matrices déjà prétraitées sont facultatives ; ne PAS les utiliser pour GridSearchCV.
EXPORTER_MATRICES_PRETRAITEES = False
if EXPORTER_MATRICES_PRETRAITEES:
    X_train_preprocessed.to_csv(dossier_sortie / 'X_train_preprocessed.csv', index=False)
    X_val_preprocessed.to_csv(dossier_sortie / 'X_val_preprocessed.csv', index=False)

print('Fichiers générés dans :', dossier_sortie)
print('Source brute inchangée :', fichier_brut)

## 8. Passer à la phase 3 : comment utiliser les fichiers sans fuite de données

Dans `03_model_training_evaluation.ipynb`, il faudra charger les fichiers `X_train_features.csv`, `y_train.csv` et `groups_train.csv`.

Le schéma à appliquer à **chacun** des six classifieurs du cours :

```python
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV

pipeline_ml = Pipeline([
    ('preparation', clone(preprocessor)),  # Imputation, encodage, standardisation
    ('modele', classifieur)               # Zero-R, KNN, arbre, RF, NB ou SVM
])

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
# GridSearchCV(..., cv=cv).fit(X_train_features, y_train, groups=groups_train)
```

Ce bloc est une **illustration** : `classifieur` et les grilles d'hyperparamètres seront définis dans le notebook 03. Le jeu `X_val_features` et `y_val` doit rester réservé à l'évaluation finale, **après** le choix des modèles.

**Bilan :** jeu divisé sans clients communs, corrections de format tracées, imputations et encodages appris seulement sur l'entraînement, validation croisée par client possible. Ne pas considérer la phase 2 validée avant d'exécuter toutes les cellules et de vérifier les sorties sur les vraies données.